<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇱🇺 Luxembourg Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: data.public.lu · Provider: Administration des transports publics · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

**Luxembourg GTFS Dataset**
- Provider: Luxembourg Open Data Portal (data.public.lu)
- Publisher: Verkéiersverbond — Luxembourg public transport authority
- Format: GTFS
- URL: https://data.public.lu/en/datasets/horaires-et-arrets-des-transport-publics-gtfs/
- Accessed: April 2026
- License: Creative Commons Attribution 4.0

**Luxembourg NeTEx Dataset**
- Provider: Luxembourg Open Data Portal (data.public.lu)
- Publisher: Verkéiersverbond — Luxembourg public transport authority
- Format: NeTEx
- URL: https://data.public.lu/en/datasets/horaires-et-arrets-des-transport-publics-netex/
- Accessed: April 2026
- License: Creative Commons Attribution 4.0

*No registration needed for downloading the datasets.

## Table of Contents

### Part 1: Luxembourg GTFS Exploration
- Load and inspect the GTFS ZIP
- Table size check
- Validate core MVD columns
- Stops: structure and data quality
- Routes: route types, short names, trips per route
- Calendar: date range, exceptions, service patterns

### Part 2: Luxembourg NeTEx Exploration
- Inspect ZIP contents and file groups
- Check main NeTEx objects across sample files
- Line extraction: test and scale
- StopPlace extraction: test and scale
- Calendar extraction: DayType, DayTypeAssignment, UicOperatingPeriod

### Part 3: Luxembourg GTFS–NeTEx Comparison
- 2.1 Station level: full parse, ID-based matching, coordinate validation
- 2.2 Line level: public code vs route short name comparison
- 2.3 Calendar level: pattern reconstruction, bit string comparison, match rate

# Part 1: Luxembourg GTFS Exploration

## Load the GTFS ZIP

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import xml.etree.ElementTree as ET
import re
from haversine import haversine, Unit
from collections import defaultdict

In [2]:
gtfs_zip_path = Path("data/gtfs-20260408-20260430.zip")
print(gtfs_zip_path)
print("Exists:", gtfs_zip_path.exists())

data/gtfs-20260408-20260430.zip
Exists: True


## Luxembourg GTFS: helper function for reading files from the ZIP

I define again a helper function to load individual GTFS text files directly from the ZIP archive.  
It is the same extraction logic as in the earlier notebooks.

In [3]:
def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

## Luxembourg GTFS: load main GTFS files

In [4]:
stops = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
stop_times = read_gtfs_from_zip(gtfs_zip_path, "stop_times.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")
agency = read_gtfs_from_zip(gtfs_zip_path, "agency.txt")

## Luxembourg GTFS: first table size check

I now check the size of the main GTFS tables after loading them.  
This gives a first overview of the Luxembourg feed and confirms that the core files were read correctly.

In [5]:
def overview(name, df):
    print(f"{name}: rows={len(df):,}, cols={df.shape[1]}")

overview("stops", stops)
overview("routes", routes)
overview("trips", trips)
overview("stop_times", stop_times)
overview("calendar", calendar)
overview("calendar_dates", calendar_dates)
overview("agency", agency)

stops: rows=2,796, cols=10
routes: rows=700, cols=8
trips: rows=30,680, cols=10
stop_times: rows=724,900, cols=8
calendar: rows=100, cols=10
calendar_dates: rows=268, cols=3
agency: rows=6, cols=6


## Output

The main Luxembourg GTFS tables were loaded successfully.  
The feed looks manageable in size, with `stop_times` as the largest table, while `calendar` and `calendar_dates` are relatively small.  

## Luxembourg GTFS: validate core columns

I now check whether the main GTFS tables contain the core columns needed for the later comparison.  
This is a simple structural check before looking at the content in more detail.

In [6]:
mvd_required = {
    "stops": ["stop_id", "stop_name", "stop_lat", "stop_lon"],
    "routes": ["route_id", "route_short_name"],
    "trips": ["trip_id", "route_id", "service_id"],
    "stop_times": ["trip_id", "stop_id", "stop_sequence"],
    "calendar": ["service_id", "start_date", "end_date"],
}

for table, cols in mvd_required.items():
    df = globals()[table]
    missing_cols = [c for c in cols if c not in df.columns]
    print(table, "missing cols:", missing_cols if missing_cols else "None")

stops missing cols: None
routes missing cols: None
trips missing cols: None
stop_times missing cols: None
calendar missing cols: None


## Comment

The main GTFS files contain the columns needed for the comparison.  

## Luxembourg GTFS: stops

I now take a first look at the stops table.  
This helps show how stop data is structured before checking station-related fields in more detail.

In [7]:
print("shape:", stops.shape)
print("columns:", list(stops.columns))
display(stops.head())

shape: (2796, 10)
columns: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code']


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
0,200403005,NaN,"Belair, Sacré-Coeur",NaN,49.610276,6.113159,0,NaN,0,NaN
1,200402003,NaN,"Beggen, Cyprien Merjai",NaN,49.643399,6.134779,0,NaN,0,NaN
2,200402004,NaN,"Beggen, Henri Dunant",NaN,49.647594,6.134546,0,NaN,0,NaN
3,200304002,NaN,"Howald, Bei der Kierch",NaN,49.583246,6.142332,0,NaN,0,NaN
4,200417037,NaN,"Kirchberg, Schmëdd",NaN,49.628363,6.149234,0,NaN,0,NaN


In [8]:
print("parent_station non-null count:", stops["parent_station"].notna().sum())
print("parent_station null count:", stops["parent_station"].isna().sum())

parent_station non-null count: 0
parent_station null count: 2796


## Observation

`parent_station` will not be used in the Luxembourg GTFS feed, since all values are empty.  
In the earlier country work, this field was only used as a helper to check whether the GTFS stop structure was hierarchical. In Austria and Switzerland, many records had a `parent_station`, which showed a station–child stop structure. Here, it is missing, so the stop hierarchy will need to be checked mainly through `location_type` instead.

## Check location_type

In [9]:
print("location_type value counts:")
print(stops["location_type"].value_counts(dropna=False))

location_type value counts:
location_type
0    2796
Name: count, dtype: int64


## Observation

All stops have `location_type = 0` in the Luxembourg GTFS feed.  
This means the stop table looks flat and does not show a station hierarchy through `location_type`.  
In the earlier country work, this field helped identify station-like records, but that does not seem possible here in the same way.

In [10]:
# Data quality check

print("Missingness in core stop fields:")
print(
    stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

Missingness in core stop fields:
stop_id      0.0
stop_name    0.0
stop_lat     0.0
stop_lon     0.0
dtype: float64


## Comment

The main stop fields are complete in the Luxembourg GTFS feed.  
So even though the stops table looks flat, the stop data is clean.

## Luxembourg GTFS: routes

In [11]:
print("shape:", routes.shape)
print("columns:", list(routes.columns))
display(routes.head())

shape: (700, 8)
columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'route_desc']


,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color,route_desc
0,4025,1,T11,NaN,3,NaN,NaN,NaN
1,4024,1,Q14,NaN,3,NaN,NaN,NaN
2,4023,1,K55,NaN,3,NaN,NaN,NaN
3,4022,1,J29,NaN,3,NaN,NaN,NaN
4,4021,1,J28,NaN,3,NaN,NaN,NaN


## Output

The routes table looks clean and structured.  
The main fields for line comparison are present, especially `route_id` and `route_short_name`.  
`route_long_name` is mostly empty in this sample, so the short name will likely be the main field to use.

## Check route short names

I now check the main route short name field more closely.  
This field was very important in the earlier country comparisons, so it is useful to see how complete it is here.

In [12]:
print("route_short_name missing:", routes["route_short_name"].isna().sum())
print("route_short_name non-missing:", routes["route_short_name"].notna().sum())
print("unique route_short_name:", routes["route_short_name"].nunique(dropna=True))

route_short_name missing: 0
route_short_name non-missing: 700
unique route_short_name: 690


## Comment

`route_short_name` is complete in the Luxembourg GTFS feed.  
Almost every route also has its own short name, with only a few repeated values.  

## Check repeated routes

As seen above the route short name field is complete, but not fully unique: there are 700 rows and 690 unique values.  
So it is useful to check which short names repeat and how often.  
In the earlier country work, the line comparison was done at the label level (after removing duplicates), not row by row. This helps decide if the same approach is needed here.

In [13]:
route_short_name_counts = (
    routes["route_short_name"]
    .value_counts(dropna=False)
    .reset_index()
)
route_short_name_counts.columns = ["route_short_name", "count"]

repeated_route_short_names = route_short_name_counts[route_short_name_counts["count"] > 1]

print("repeated route_short_name values:", len(repeated_route_short_names))
display(repeated_route_short_names.head(20))

repeated route_short_name values: 10


,route_short_name,count
0,13,2
1,14,2
2,2,2
3,3,2
4,4,2
5,5,2
6,6,2
7,7,2
8,15,2
9,A20,2


## Comment

Only a small number of route short names are repeated, and each of them appears only twice.  
So most routes have their own unique label, and duplicates are very limited.  
This means the line data looks clean and can likely be used directly or with minimal deduplication.

## Check route types

I now check the `route_type` values in the routes table.  
This shows which kinds of transport are present in the Luxembourg GTFS feed.

In [14]:
print("route_type value counts:")
print(routes["route_type"].value_counts(dropna=False).sort_index())

route_type value counts:
route_type
0      1
2      5
3    694
Name: count, dtype: int64


## Observation

Most routes have `route_type = 3`, which means they are **bus routes**.
There are only a few routes of other types.
So the Luxembourg GTFS feed is mainly focused on **bus transport**.

---

### GTFS Route Type Codes

| Code | Type | Description |
|------|------|-------------|
| `0` | Tram / Streetcar / Light Rail | Any light rail or street level system within a metropolitan area |
| `1` | Subway / Metro | Any underground rail system within a metropolitan area |
| `2` | Rail | Used for intercity or long-distance travel |
| `3` | Bus | Used for short- and long-distance bus routes |
| `4` | Ferry | Used for short- and long-distance boat service |
| `5` | Cable Tram | Street-level rail cars where the cable runs beneath the vehicle *(e.g., cable car in San Francisco)* |
| `6` | Aerial Lift / Suspended Cable Car | Cable transport where cabins, cars, gondolas or open chairs are suspended by one or more cables *(e.g., gondola lift, aerial tramway)* |
| `7` | Funicular | Any rail system designed for steep inclines |
| `11` | Trolleybus | Electric buses that draw power from overhead wires using poles |
| `12` | Monorail | Railway in which the track consists of a single rail or a beam |

> Source: [gtfs.org — Schedule Reference](https://gtfs.org/documentation/schedule/reference/)

## Check trips per route

I now check how many trips are linked to each route.  
This helps to see whether a few routes dominate the feed or whether trips are more spread out.

In [15]:
trips_per_route = (
    trips.groupby("route_id")
    .size()
    .reset_index(name="n_trips")
    .sort_values("n_trips", ascending=False)
)

print("number of routes in trips:", trips_per_route["route_id"].nunique())
display(trips_per_route.head(20))

number of routes in trips: 700


,route_id,n_trips
103,2466,1388
24,299,1130
8,265,705
10,267,663
13,270,602
4,261,580
42,1967,566
22,279,551
5,262,524
1,258,516


## Observation

All routes are used in the trips table, and the number of trips per route varies a lot.  
Some routes have many trips, while others have fewer.  
This shows that the route table is well connected to the service data.

## Luxembourg GTFS: calendar

In [16]:
print("shape:", calendar.shape)
print("columns:", list(calendar.columns))
display(calendar.head())

shape: (100, 10)
columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20260408,20260430
1,2,0,1,0,1,0,0,0,20260408,20260430
2,3,1,0,1,0,1,0,0,20260408,20260430
3,4,0,0,1,1,1,0,0,20260408,20260430
4,5,0,1,1,1,0,0,0,20260408,20260430


## Observation

The first few rows look clean and complete. 
The table includes the standard weekly service pattern and a clear start and end date for each service.  
This means the service validity structure is easy to work with.

## Look at calendar_dates

I now take a first look at the calendar_dates table.  
This table shows service exceptions, such as added or removed dates.

In [17]:
print("shape:", calendar_dates.shape)
print("columns:", list(calendar_dates.columns))
display(calendar_dates.head())

shape: (268, 3)
columns: ['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,1,20260408,2
1,1,20260409,2
2,1,20260410,2
3,2,20260409,2
4,3,20260408,2


## Observation

The calendar_dates table looks clean and simple.  
It contains service exceptions with a date and an exception type for each service.  
This means it can be combined easily with the calendar table.

In [18]:
# Check exception_type

print("exception_type value counts:")
print(calendar_dates["exception_type"].value_counts(dropna=False).sort_index())

exception_type value counts:
exception_type
1    103
2    165
Name: count, dtype: int64


## Observation

Both exception types are present in the Luxembourg GTFS feed, which means the dataset includes both added and removed service days.  
So the full service validity can be reconstructed using calendar and calendar_dates together.

## Check overall date range

In [19]:
print("calendar start_date min:", calendar["start_date"].min())
print("calendar end_date max:", calendar["end_date"].max())
print("calendar_dates date min:", calendar_dates["date"].min())
print("calendar_dates date max:", calendar_dates["date"].max())
print("unique service_id in calendar:", calendar["service_id"].nunique())
print("unique service_id in calendar_dates:", calendar_dates["service_id"].nunique())

calendar start_date min: 20260408
calendar end_date max: 20260430
calendar_dates date min: 20260408
calendar_dates date max: 20260430
unique service_id in calendar: 100
unique service_id in calendar_dates: 87


## Observation

The Luxembourg GTFS feed covers the period from 2026-04-08 to 2026-04-30.  
The date range is the same in both `calendar` and `calendar_dates`.  
So the GTFS service validity period is clear and consistent.

# Summary: Luxembourg GTFS

The Luxembourg GTFS feed looks clean and manageable.  
The stops table is flat, because `parent_station` is empty and all records have `location_type = 0`.  
Still, the main stop fields are complete, so the stop data can be used for later comparison.  

On the line side, the routes table is clean and `route_short_name` is complete, with only a few repeated labels.  


On the calendar side, both `calendar` and `calendar_dates` are simple and consistent.  
The GTFS feed covers the period from 2026-04-08 to 2026-04-30.

# Part 2: Luxembourg Netex Exploration

In [20]:
netex_zip_path = Path("data/netex-20260408-20260430.zip")
print(netex_zip_path)
print("Exists:", netex_zip_path.exists())

data/netex-20260408-20260430.zip
Exists: True


## Inspect ZIP contents

In [21]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

netex_files = pd.DataFrame([
    {
        "name": i.filename,
        "size_mb": round(i.file_size / (1024 * 1024), 2)
    }
    for i in infos
])

display(netex_files.sort_values("size_mb", ascending=False).head(30))
print("total files:", len(netex_files))
print("xml files:", netex_files["name"].str.lower().str.endswith(".xml").sum())

,name,size_mb
699,TRAM_TRAM/NX-PI-01_LU_NAP_LINE_TRAM-TRAM-T1_20...,15.98
689,TICE_TIC/NX-PI-01_LU_NAP_LINE_TICE-TIC-01_2026...,10.17
692,TICE_TIC/NX-PI-01_LU_NAP_LINE_TICE-TIC-04_2026...,8.97
1,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-11_202604...,8.56
693,TICE_TIC/NX-PI-01_LU_NAP_LINE_TICE-TIC-05_2026...,8.21
19,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-29_202604...,8.11
691,TICE_TIC/NX-PI-01_LU_NAP_LINE_TICE-TIC-03_2026...,7.99
690,TICE_TIC/NX-PI-01_LU_NAP_LINE_TICE-TIC-02_2026...,7.77
8,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-18_202604...,7.77
10,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-2_2026040...,7.63


total files: 700
xml files: 700


## Comment

The Luxembourg NeTEx dataset contains many XML files, with one file per line.  
The structure looks fragmented, similar to the Germany case, rather than a small number of grouped files like in Switzerland.  

## Check file groups

I now check how the NeTEx XML files are grouped inside the ZIP file.  
This helps show whether the dataset is organised in clear sections before opening individual XML files.

In [22]:
netex_files["group"] = netex_files["name"].str.split("/").str[0]

group_counts = (
    netex_files["group"]
    .value_counts()
    .reset_index()
)
group_counts.columns = ["group", "n_files"]

display(group_counts)

,group,n_files
0,RGTR_RGTR,608
1,AVL_AVL,57
2,CFL_B_CFLB,19
3,TICE_TIC,10
4,CFL_CFL82,5
5,TRAM_TRAM,1


## Comment

The Luxembourg NeTEx dataset is split into several clear file groups.  
Most files belong to `RGTR_RGTR`, while the other groups are much smaller.  
So the dataset looks fragmented, but the fragmentation is organised in a clear way.

## Inspect one XML file from the largest group

I now open one XML file from the largest NeTEx group.  
This gives a first look at the internal XML structure before checking more files.

In [23]:
sample_xml = (
    netex_files[netex_files["group"] == "RGTR_RGTR"]
    .sort_values("name")
    .iloc[0]["name"]
)

print("Opening:", sample_xml)

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        head = f.read(4000)

print(head.decode("utf-8", errors="replace"))

Opening: RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_20260408.xml
<?xml version="1.0" encoding="UTF-8" standalone="no" ?>
<PublicationDelivery xmlns="http://www.netex.org.uk/netex" version="ntx:1.1">

  <PublicationTimestamp>2026-04-08T10:13:38</PublicationTimestamp>

  <ParticipantRef>LU</ParticipantRef>

  <PublicationRequest>
    <RequestTimestamp>2026-04-08T10:13:38</RequestTimestamp>
    <topics>
      <NetworkFrameTopic>
        <Current/>
        <NetworkFilterByValue>
          <objectReferences>
            <LineRef ref="LU::Line:3741::" version="1775636018"/>
            <TypeOfFrameRef ref="epip:EU_PI_LINE_OFFER" versionRef="1.0"/>
          </objectReferences>
        </NetworkFilterByValue>
      </NetworkFrameTopic>
    </topics>
  </PublicationRequest>

  <dataObjects>
    <CompositeFrame id="LU::CompositeFrame_EU_PI_LINE_OFFER:RGTR-RGTR-110" version="1775636018">
      <ValidBetween>
        <FromDate>2026-04-08T00:00:00</FromDate>
        <ToDate>2026-04-30T00:00:00</

## Check main objects in this sample file

I now check whether the sample NeTEx file contains the main objects needed for the later comparison.  
This gives a quick first idea of whether the file includes line, stop, and calendar-related information.

In [24]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        xml_text = f.read().decode("utf-8", errors="replace")

tags_to_check = [
    "StopPlace",
    "ScheduledStopPoint",
    "Line",
    "Route",
    "ServiceJourney",
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "UicOperatingPeriod"
]

for tag in tags_to_check:
    print(f"{tag}:", f"<{tag}" in xml_text)

StopPlace: True
ScheduledStopPoint: True
Line: True
Route: True
ServiceJourney: True
DayType: True
DayTypeAssignment: True
OperatingPeriod: True
UicOperatingPeriod: True


## Observation

This sample NeTEx file already contains the main objects needed for the three comparison levels.  
It includes stop-related objects (`StopPlace`, `ScheduledStopPoint`), line-related objects (`Line`, `Route`, `ServiceJourney`), and calendar-related objects (`DayType`, `DayTypeAssignment`, `OperatingPeriod`, `UicOperatingPeriod`).  
So this is a good sign: at least in this first sample, the Luxembourg NeTEx structure seems rich enough for the same kind of comparison used before.

## Check main objects across a small file sample

I now check whether the same main NeTEx objects appear in several files from the largest group.  
This helps show whether the structure seen in the first sample is stable.

In [25]:
sample_files = (
    netex_files[netex_files["group"] == "RGTR_RGTR"]
    .sort_values("name")
    .head(10)["name"]
    .tolist()
)

tags_to_check = [
    "StopPlace",
    "ScheduledStopPoint",
    "Line",
    "Route",
    "ServiceJourney",
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "UicOperatingPeriod"
]

results = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            xml_text = f.read().decode("utf-8", errors="replace")

        row = {"file_name": file_name}
        for tag in tags_to_check:
            row[tag] = f"<{tag}" in xml_text
        results.append(row)

sample_tag_check = pd.DataFrame(results)
display(sample_tag_check)

,file_name,StopPlace,ScheduledStopPoint,Line,Route,ServiceJourney,DayType,DayTypeAssignment,OperatingPeriod,UicOperatingPeriod
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,True,True,True,True,True,True,True,True,True
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,True,True,True,True,True,True,True,True,True
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,True,True,True,True,True,True,True,True,True
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-113_2...,True,True,True,True,True,True,True,True,True
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-114_2...,True,True,True,True,True,True,True,True,True
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-115_2...,True,True,True,True,True,True,True,True,True
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-116_2...,True,True,True,True,True,True,True,True,True
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-117_2...,True,True,True,True,True,True,True,True,True
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-118_2...,True,True,True,True,True,True,True,True,True
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-119_2...,True,True,True,True,True,True,True,True,True


## Comment

The same main NeTEx objects appear in all 10 sampled files from the largest group.  
So the structure does not seem to change from file to file in this first sample.  

## Test line extraction on one sample file

I now test a first line extraction on one sample NeTEx file.  
This shows whether the main line fields can be read in a clean way before applying the same logic to more files.

In [26]:
# allows to query elements with the "n:" prefix
ns = {"n": "http://www.netex.org.uk/netex"}

def child_text(elem, tag):
    # Retrieve the text of a direct child element, returning None if missing
    child = elem.find(f"n:{tag}", ns)
    return child.text.strip() if child is not None and child.text is not None else None

# Open the ZIP and parse the sample NeTEx XML file into an element tree
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        tree = ET.parse(f)

root = tree.getroot()

# Search the entire XML tree for <Line> elements and extract key fields
line_rows = []
for line in root.findall(".//n:Line", ns):
    line_rows.append({
        "line_id":        line.get("id"),                      # XML attribute
        "line_name":      child_text(line, "Name"),            # Child elements
        "short_name":     child_text(line, "ShortName"),
        "public_code":    child_text(line, "PublicCode"),
        "transport_mode": child_text(line, "TransportMode"),
    })

# Convert to a DataFrame and preview the first 20 rows
sample_lines = pd.DataFrame(line_rows)
print("number of Line objects:", len(sample_lines))
display(sample_lines.head(20))

number of Line objects: 1


,line_id,line_name,short_name,public_code,transport_mode
0,LU::Line:3741::,110,110,110,bus


## Comment

The first line extraction works well on the sample file.  
The file contains one clear line object, and the main fields can be read without problems.  
In this sample, the visible line label is `110`, and the transport mode is `bus`.  

## Test line extraction across a small file sample

I now apply the same line extraction to a small sample of files from the largest group.  

In [27]:
sample_line_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            tree = ET.parse(f)

        root = tree.getroot()

        for line in root.findall(".//n:Line", ns):
            sample_line_rows.append({
                "file_name": file_name,
                "line_id": line.get("id"),
                "line_name": child_text(line, "Name"),
                "short_name": child_text(line, "ShortName"),
                "public_code": child_text(line, "PublicCode"),
                "transport_mode": child_text(line, "TransportMode"),
            })

sample_lines_multi = pd.DataFrame(sample_line_rows)

print("number of extracted lines:", len(sample_lines_multi))
display(sample_lines_multi.head(20))

number of extracted lines: 10


,file_name,line_id,line_name,short_name,public_code,transport_mode
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::Line:3741::,110,110,110,bus
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::Line:3167::,111,111,111,bus
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,LU::Line:3168::,112,112,112,bus
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-113_2...,LU::Line:3742::,113,113,113,bus
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-114_2...,LU::Line:3743::,114,114,114,bus
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-115_2...,LU::Line:3169::,115,115,115,bus
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-116_2...,LU::Line:2327::,116,116,116,bus
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-117_2...,LU::Line:2328::,117,117,117,bus
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-118_2...,LU::Line:3744::,118,118,118,bus
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-119_2...,LU::Line:2391::,119,119,119,bus


## Comment

The same line extraction works across all 10 sampled files.  
In each file, the main line fields are filled and easy to read.  
In this sample, the visible line label matches `line_name`, `short_name`, and `public_code`, and all extracted lines are bus lines.  
So the Luxembourg NeTEx line structure looks very consistent in this first sample.

## Scale line extraction to a larger file sample

I now apply the same line extraction to a larger sample of files.
This shows whether the same line structure stays consistent beyond the first small sample.

In [28]:
large_sample_files = (
    netex_files[netex_files["group"] == "RGTR_RGTR"]
    .sort_values("name")
    .head(100)["name"]
    .tolist()
)

large_line_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in large_sample_files:
        with z.open(file_name) as f:
            tree = ET.parse(f)

        root = tree.getroot()

        for line in root.findall(".//n:Line", ns):
            large_line_rows.append({
                "file_name": file_name,
                "line_id": line.get("id"),
                "line_name": child_text(line, "Name"),
                "short_name": child_text(line, "ShortName"),
                "public_code": child_text(line, "PublicCode"),
                "transport_mode": child_text(line, "TransportMode"),
            })

large_lines = pd.DataFrame(large_line_rows)

print("sampled files:", len(large_sample_files))
print("extracted lines:", len(large_lines))
print("unique public_code:", large_lines["public_code"].nunique(dropna=True))
display(large_lines.head(20))

sampled files: 100
extracted lines: 100
unique public_code: 100


,file_name,line_id,line_name,short_name,public_code,transport_mode
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::Line:3741::,110,110,110,bus
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::Line:3167::,111,111,111,bus
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,LU::Line:3168::,112,112,112,bus
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-113_2...,LU::Line:3742::,113,113,113,bus
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-114_2...,LU::Line:3743::,114,114,114,bus
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-115_2...,LU::Line:3169::,115,115,115,bus
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-116_2...,LU::Line:2327::,116,116,116,bus
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-117_2...,LU::Line:2328::,117,117,117,bus
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-118_2...,LU::Line:3744::,118,118,118,bus
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-119_2...,LU::Line:2391::,119,119,119,bus


## Comment

The larger sample gives the same result as the smaller one.  
In all 100 sampled files, one line was extracted per file, and all 100 `public_code` values are unique.  
The main line fields are filled in a very consistent way, and the visible line label matches `line_name`, `short_name`, and `public_code`.  

## Test StopPlace extraction on one sample file

I now test a first extraction of `StopPlace` objects from the same sample NeTEx file.  
This helps show how stop and station information is represented before checking more files.

In [29]:
def find_text(elem, path):
    child = elem.find(path, ns)
    return child.text.strip() if child is not None and child.text is not None else None

stopplace_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        tree = ET.parse(f)

root = tree.getroot()

for sp in root.findall(".//n:StopPlace", ns):
    stopplace_rows.append({
        "stopplace_id": sp.get("id"),
        "stopplace_name": find_text(sp, "n:Name"),
        "public_code": find_text(sp, "n:PublicCode"),
        "latitude": find_text(sp, ".//n:Latitude"),
        "longitude": find_text(sp, ".//n:Longitude"),
    })

sample_stopplaces = pd.DataFrame(stopplace_rows)

print("number of StopPlace objects:", len(sample_stopplaces))
display(sample_stopplaces.head(20))

number of StopPlace objects: 31


,stopplace_id,stopplace_name,public_code,latitude,longitude
0,LU::StopPlace:200417047_CdT::,"Kirchberg, Rout Bréck - Pafendall (Bus)",None,49.618368,6.13575
1,LU::StopPlace:200405014_CdT::,"Centre, Fondation Pescatore",None,49.615707,6.126753
2,LU::StopPlace:160904001_CdT::,"Mersch, Gare routière",None,49.753054,6.110115
3,LU::StopPlace:160807003_CdT::,"Lorentzweiler, Gare routière",None,49.698348,6.140937
4,LU::StopPlace:200409007_CdT::,"Dommeldange, Emile Metz",None,49.635082,6.134256
5,LU::StopPlace:200410004_CdT::,"Eich, Eecher Klinick",None,49.63179,6.134643
6,LU::StopPlace:200410003_CdT::,"Eich, Eecher Plaz",None,49.627562,6.129933
7,LU::StopPlace:160702003_CdT::,"Lintgen, Kräizung",None,49.720348,6.125116
8,LU::StopPlace:200419013_CdT::,"Limpertsberg, Crispinus",None,49.619347,6.130236
9,LU::StopPlace:200802004_CdT::,"Heisdorf, Klouster",None,49.674509,6.137331


## Comment

The sample file contains many `StopPlace` objects, and the main fields can be read without problems.  
The stop names and coordinates are filled, but `public_code` is empty in this sample.  

## Test StopPlace extraction across a small file sample

I now apply the same StopPlace extraction to a small sample of files from the largest group.  

In [30]:
sample_stopplace_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            tree = ET.parse(f)

        root = tree.getroot()

        for sp in root.findall(".//n:StopPlace", ns):
            sample_stopplace_rows.append({
                "file_name": file_name,
                "stopplace_id": sp.get("id"),
                "stopplace_name": find_text(sp, "n:Name"),
                "public_code": find_text(sp, "n:PublicCode"),
                "latitude": find_text(sp, ".//n:Latitude"),
                "longitude": find_text(sp, ".//n:Longitude"),
            })

sample_stopplaces_multi = pd.DataFrame(sample_stopplace_rows)

print("sampled files:", len(sample_files))
print("extracted StopPlace objects:", len(sample_stopplaces_multi))
print("files with at least one StopPlace:", sample_stopplaces_multi["file_name"].nunique())
display(sample_stopplaces_multi.head(20))

sampled files: 10
extracted StopPlace objects: 309
files with at least one StopPlace: 10


,file_name,stopplace_id,stopplace_name,public_code,latitude,longitude
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200417047_CdT::,"Kirchberg, Rout Bréck - Pafendall (Bus)",None,49.618368,6.13575
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200405014_CdT::,"Centre, Fondation Pescatore",None,49.615707,6.126753
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:160904001_CdT::,"Mersch, Gare routière",None,49.753054,6.110115
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:160807003_CdT::,"Lorentzweiler, Gare routière",None,49.698348,6.140937
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200409007_CdT::,"Dommeldange, Emile Metz",None,49.635082,6.134256
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200410004_CdT::,"Eich, Eecher Klinick",None,49.63179,6.134643
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200410003_CdT::,"Eich, Eecher Plaz",None,49.627562,6.129933
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:160702003_CdT::,"Lintgen, Kräizung",None,49.720348,6.125116
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200419013_CdT::,"Limpertsberg, Crispinus",None,49.619347,6.130236
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::StopPlace:200802004_CdT::,"Heisdorf, Klouster",None,49.674509,6.137331


## Comment

The StopPlace extraction also works across all 10 sampled files.  

Each sampled file contains StopPlace objects, and the stop names and coordinates are filled in a consistent way.  

At the same time, `public_code` is still empty in this sample.  


## Check quality of the main StopPlace fields

I now check whether the main StopPlace fields are complete in the sampled files.  

In [31]:
print("missing stopplace_name:", sample_stopplaces_multi["stopplace_name"].isna().sum())
print("missing public_code:", sample_stopplaces_multi["public_code"].isna().sum())
print("missing latitude:", sample_stopplaces_multi["latitude"].isna().sum())
print("missing longitude:", sample_stopplaces_multi["longitude"].isna().sum())
print("unique StopPlace ids:", sample_stopplaces_multi["stopplace_id"].nunique())

missing stopplace_name: 0
missing public_code: 309
missing latitude: 0
missing longitude: 0
unique StopPlace ids: 141


## Comment

The main StopPlace fields are very complete in this sample.  

All StopPlace objects have a name and coordinates, but `public_code` is missing everywhere.  

Also, the number of unique StopPlace ids is much smaller than the total number of extracted rows.  

This means many StopPlace objects appear again in different files, so later stop work will likely need deduplication. 

For the Netex dataset, stop name, coordinates, and StopPlace id currently look more useful than `public_code`.

## Test calendar extraction on one sample file

I now test a first extraction of the main calendar-related objects from the same sample file.  

In [32]:
def get_ref(elem, path):
    child = elem.find(path, ns)
    return child.get("ref") if child is not None else None

daytype_rows = []
daytypeassignment_rows = []
operatingperiod_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        tree = ET.parse(f)

root = tree.getroot()

for dt in root.findall(".//n:DayType", ns):
    daytype_rows.append({
        "daytype_id": dt.get("id"),
        "daytype_name": find_text(dt, "n:Name"),
    })

for dta in root.findall(".//n:DayTypeAssignment", ns):
    daytypeassignment_rows.append({
        "assignment_id": dta.get("id"),
        "order": find_text(dta, "n:Order"),
        "date": find_text(dta, "n:Date"),
        "daytype_ref": get_ref(dta, "n:DayTypeRef"),
        "operating_period_ref": get_ref(dta, "n:OperatingPeriodRef"),
    })

for op in root.findall(".//n:UicOperatingPeriod", ns):
    operatingperiod_rows.append({
        "operating_period_id": op.get("id"),
        "from_date": find_text(op, "n:FromDate"),
        "to_date": find_text(op, "n:ToDate"),
        "valid_day_bits": find_text(op, "n:ValidDayBits"),
    })

sample_daytypes = pd.DataFrame(daytype_rows)
sample_daytypeassignments = pd.DataFrame(daytypeassignment_rows)
sample_operatingperiods = pd.DataFrame(operatingperiod_rows)

print("number of DayType objects:", len(sample_daytypes))
display(sample_daytypes.head(10))

print("number of DayTypeAssignment objects:", len(sample_daytypeassignments))
display(sample_daytypeassignments.head(10))

print("number of UicOperatingPeriod objects:", len(sample_operatingperiods))
display(sample_operatingperiods.head(10))

number of DayType objects: 3


,daytype_id,daytype_name
0,LU::DayType:645::,None
1,LU::DayType:646::,None
2,LU::DayType:647::,None


number of DayTypeAssignment objects: 3


,assignment_id,order,date,daytype_ref,operating_period_ref
0,LU::DayTypeAssignment:645::,None,None,LU::DayType:645::,LU::UicOperatingPeriod:645::
1,LU::DayTypeAssignment:646::,None,None,LU::DayType:646::,LU::UicOperatingPeriod:646::
2,LU::DayTypeAssignment:647::,None,None,LU::DayType:647::,LU::UicOperatingPeriod:647::


number of UicOperatingPeriod objects: 3


,operating_period_id,from_date,to_date,valid_day_bits
0,LU::UicOperatingPeriod:645::,2026-04-08T00:00:00,2026-04-30T00:00:00,11100111110011111001111
1,LU::UicOperatingPeriod:646::,2026-04-08T00:00:00,2026-04-30T00:00:00,11110111111011111101111
2,LU::UicOperatingPeriod:647::,2026-04-11T00:00:00,2026-04-25T00:00:00,100000010000001


## Output

The sample file contains the main calendar-related objects needed for service validity.  
`DayTypeAssignment` links the day types to `UicOperatingPeriod`, and the operating periods contain clear date ranges and `valid_day_bits`.  
So the calendar structure looks usable and is similar to the calendar logic used before.

## Test calendar extraction across a small file sample

I now apply the same calendar extraction to a small sample of files from the largest group.  


In [33]:
calendar_sample_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            tree = ET.parse(f)

        root = tree.getroot()

        n_daytypes = len(root.findall(".//n:DayType", ns))
        n_daytypeassignments = len(root.findall(".//n:DayTypeAssignment", ns))
        n_uicoperatingperiods = len(root.findall(".//n:UicOperatingPeriod", ns))

        calendar_sample_rows.append({
            "file_name": file_name,
            "n_daytypes": n_daytypes,
            "n_daytypeassignments": n_daytypeassignments,
            "n_uicoperatingperiods": n_uicoperatingperiods,
        })

calendar_sample_check = pd.DataFrame(calendar_sample_rows)
display(calendar_sample_check)

,file_name,n_daytypes,n_daytypeassignments,n_uicoperatingperiods
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,3,3,3
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,5,5,5
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,5,5,5
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-113_2...,4,4,4
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-114_2...,3,3,3
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-115_2...,3,3,3
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-116_2...,5,5,5
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-117_2...,3,3,3
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-118_2...,3,3,3
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-119_2...,5,5,5


## Comment

The same calendar structure appears in all 10 sampled files.  
Each file contains `DayType`, `DayTypeAssignment`, and `UicOperatingPeriod` objects.  

## Check operating period fields in the sample

I now check the main fields inside the sampled `UicOperatingPeriod` objects.  
This shows whether the calendar structure is filled in a useful way.

In [34]:
operatingperiod_sample_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            tree = ET.parse(f)

        root = tree.getroot()

        for op in root.findall(".//n:UicOperatingPeriod", ns):
            operatingperiod_sample_rows.append({
                "file_name": file_name,
                "operating_period_id": op.get("id"),
                "from_date": find_text(op, "n:FromDate"),
                "to_date": find_text(op, "n:ToDate"),
                "valid_day_bits": find_text(op, "n:ValidDayBits"),
            })

operatingperiod_sample = pd.DataFrame(operatingperiod_sample_rows)

print("sampled files:", len(sample_files))
print("extracted UicOperatingPeriod objects:", len(operatingperiod_sample))
print("missing from_date:", operatingperiod_sample["from_date"].isna().sum())
print("missing to_date:", operatingperiod_sample["to_date"].isna().sum())
print("missing valid_day_bits:", operatingperiod_sample["valid_day_bits"].isna().sum())

display(operatingperiod_sample.head(20))

sampled files: 10
extracted UicOperatingPeriod objects: 39
missing from_date: 0
missing to_date: 0
missing valid_day_bits: 0


,file_name,operating_period_id,from_date,to_date,valid_day_bits
0,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::UicOperatingPeriod:645::,2026-04-08T00:00:00,2026-04-30T00:00:00,11100111110011111001111
1,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::UicOperatingPeriod:646::,2026-04-08T00:00:00,2026-04-30T00:00:00,11110111111011111101111
2,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-110_2...,LU::UicOperatingPeriod:647::,2026-04-11T00:00:00,2026-04-25T00:00:00,100000010000001
3,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::UicOperatingPeriod:1324::,2026-04-08T00:00:00,2026-04-30T00:00:00,11110111111011111101111
4,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::UicOperatingPeriod:1325::,2026-04-08T00:00:00,2026-04-30T00:00:00,11111111111111111111111
5,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::UicOperatingPeriod:1326::,2026-04-11T00:00:00,2026-04-25T00:00:00,100000010000001
6,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::UicOperatingPeriod:1327::,2026-04-11T00:00:00,2026-04-26T00:00:00,1100000110000011
7,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-111_2...,LU::UicOperatingPeriod:1328::,2026-04-08T00:00:00,2026-04-30T00:00:00,11100111110011111001111
8,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,LU::UicOperatingPeriod:1319::,2026-04-08T00:00:00,2026-04-30T00:00:00,11111111111111111111111
9,RGTR_RGTR/NX-PI-01_LU_NAP_LINE_RGTR-RGTR-112_2...,LU::UicOperatingPeriod:1320::,2026-04-08T00:00:00,2026-04-30T00:00:00,11100111110011111001111


## Comment

The main operating period fields are complete in this sample.  
All sampled `UicOperatingPeriod` objects have a start date, an end date, and `valid_day_bits`.  


## Check sampled date range

I now summarise the overall date range of the sampled NeTEx operating periods.  


In [35]:
print("operating_period_start_min:", operatingperiod_sample["from_date"].min())
print("operating_period_end_max:", operatingperiod_sample["to_date"].max())
print("n_operating_period_rows:", len(operatingperiod_sample))
print("n_unique_operating_period_ids:", operatingperiod_sample["operating_period_id"].nunique())

operating_period_start_min: 2026-04-08T00:00:00
operating_period_end_max: 2026-04-30T00:00:00
n_operating_period_rows: 39
n_unique_operating_period_ids: 39


The sampled NeTEx operating periods cover the date range seen earlier from 2026-04-08 to 2026-04-30.   
Also, all sampled operating period ids are unique.

## Summary: Luxembourg NeTEx

The Netex dataset is split into many XML files, mainly in the `RGTR_RGTR` group.  
So the dataset is fragmented, but the file grouping is clear.

On the **line side**, the first extraction tests worked very well.  
The same line fields were present across the sampled files, and the visible line label matched `line_name`, `short_name`, and `public_code`.  
In the sampled `RGTR_RGTR` files, the extracted lines were bus lines and the structure was very consistent.

On the **stop side**, `StopPlace` extraction also worked across the sampled files.  
The main useful fields were `stopplace_name`, coordinates, and `stopplace_id`.  
`public_code` was empty in this sample, and many StopPlace objects appeared again in different files, so later stop work will likely need deduplication.

On the **calendar side**, the sampled files all contained `DayType`, `DayTypeAssignment`, and `UicOperatingPeriod`.  
The main operating period fields were filled consistently, including start date, end date, and `valid_day_bits`.  
The sampled NeTEx date range was 2026-04-08 to 2026-04-30, which matches the Luxembourg GTFS date range.

# Part 3: Luxembourg GTFS–NeTEx Comparison

## 2.1 Station level


I start with the station level, following the same order as in the earlier notebooks.  
In Luxembourg, the GTFS stops table does not show a clear station hierarchy, because `parent_station` is empty and all records have `location_type = 0`.  
So the first step is to define the GTFS stop-side unit for the station-level analysis.

In [36]:
gtfs_station_side = stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]].copy()

print("GTFS station-side rows:", len(gtfs_station_side))
display(gtfs_station_side.head())

GTFS station-side rows: 2796


,stop_id,stop_name,stop_lat,stop_lon
0,200403005,"Belair, Sacré-Coeur",49.610276,6.113159
1,200402003,"Beggen, Cyprien Merjai",49.643399,6.134779
2,200402004,"Beggen, Henri Dunant",49.647594,6.134546
3,200304002,"Howald, Bei der Kierch",49.583246,6.142332
4,200417037,"Kirchberg, Schmëdd",49.628363,6.149234


## Comment

For Luxembourg, the full GTFS stops table is used as the stop-side unit.  
This is different from the earlier cases, where a smaller station-like subset could be defined more clearly.  
Here, the stop-side analysis starts from all GTFS stop records with name and coordinates. 
In other words, there is no clear station-only subset to filter first like I did in the earlier cases.  

## Comment

The sampled NeTEx stop-side table is ready for the station-level analysis.  
It contains StopPlace id, name, and coordinates for 309 rows from the sampled files.  
Unlike the GTFS side, this NeTEx sample still includes repeated StopPlace objects across different files (NeTEx stop-side sample has 309 rows but only 141 unique stopplace_id values).

## Step 1: Parse all Netex files to get the full StopPlace table

The NeTEx feed is not one big file, it's split into 700 individual XML files, one per route line. Each file contains the full description of that line, including all the stops it serves. To get a complete picture of all stations, I had to open and read every single one of those files. For each file, I searched the XML tree for <StopPlace> elements and extracted the stop ID, name, latitude, and longitude. Because the same physical stop is served by many lines, the same StopPlace appeared repeatedly across different files giving 16,600 raw rows. I then deduplicated on stopplace_id to keep only one entry per unique station.

In [37]:
all_stopplace_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 100 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for sp in root.findall(".//n:StopPlace", ns):
            all_stopplace_rows.append({
                "file_name":      file_name,
                "stopplace_id":   sp.get("id"),
                "stopplace_name": find_text(sp, "n:Name"),
                "latitude":       find_text(sp, ".//n:Latitude"),
                "longitude":      find_text(sp, ".//n:Longitude"),
            })

# Raw table before deduplication
netex_all_raw = pd.DataFrame(all_stopplace_rows)

# Deduplicate (same StopPlace appears in many route files)
netex_all = netex_all_raw.drop_duplicates(subset=["stopplace_id"]).copy()

print(f"\nRaw StopPlace rows (with duplicates): {len(netex_all_raw):,}")
print(f"Unique StopPlaces after dedup:        {len(netex_all):,}")
print(f"Missing latitude:  {netex_all['latitude'].isna().sum()}")
print(f"Missing longitude: {netex_all['longitude'].isna().sum()}")

Total XML files to parse: 700
  0/700 files processed...
  100/700 files processed...
  200/700 files processed...
  300/700 files processed...
  400/700 files processed...
  500/700 files processed...
  600/700 files processed...

Raw StopPlace rows (with duplicates): 16,600
Unique StopPlaces after dedup:        2,798
Missing latitude:  0
Missing longitude: 0


## Comment

After deduplication: 2,798 unique NeTEx StopPlaces, almost exactly the same as the 2,796 GTFS stops

There are zero missing coordinates on the NeTEx side.
This near-perfect match in count is a very encouraging sign that both datasets are describing the same network.  

## Extract numeric stop ID

Each NeTEx stopplace_id follows the pattern LU::StopPlace:201004002_CdT::, where the numeric part in the middle is the same identifier used as stop_id in GTFS. I used a regex to extract that number from every NeTEx ID, then joined the two tables on it  with GTFS stop_id on the left, extracted NeTEx ID on the right. Then I  used a left join, meaning every GTFS stop is kept regardless of whether a NeTEx match was found, and unmatched ones simply get NaN in the NeTEx columns.

In [38]:
# Extract the numeric stop ID embedded in the NeTEx stopplace_id
# e.g. "LU::StopPlace:201004002_CdT::" → "201004002"
netex_all["extracted_stop_id"] = (
    netex_all["stopplace_id"]
    .str.extract(r":(\d+)_")
)

print("Sample of extracted IDs:")
print(netex_all[["stopplace_id", "extracted_stop_id"]].head(10).to_string(index=False))
print(f"\nMissing extracted_stop_id: {netex_all['extracted_stop_id'].isna().sum()}")

Sample of extracted IDs:
                 stopplace_id extracted_stop_id
LU::StopPlace:200405020_CdT::         200405020
LU::StopPlace:200415009_CdT::         200415009
LU::StopPlace:200415017_CdT::         200415017
LU::StopPlace:200405036_CdT::         200405036
LU::StopPlace:200409007_CdT::         200409007
LU::StopPlace:200410004_CdT::         200410004
LU::StopPlace:200101002_CdT::         200101002
LU::StopPlace:200410003_CdT::         200410003
LU::StopPlace:200101007_CdT::         200101007
LU::StopPlace:200419013_CdT::         200419013

Missing extracted_stop_id: 0


## Comment

The ID extraction worked perfectly with 0 missing values, meaning every single NeTEx StopPlace had a recognisable numeric ID I could extract. The extracted IDs match the format of GTFS stop_id exactly (e.g. 200405020), so no reformatting was needed.

## Join the two datasets

I merged the GTFS stops table with the NeTEx StopPlace table using the extracted numeric ID as the join key (stop_id on the GTFS side, extracted_stop_id on the NeTEx side). I used a left join so that every GTFS stop was kept in the result, with NeTEx columns filled in where a match was found and left as NaN where not.

In [39]:
# Ensure both ID columns are strings for a clean join
gtfs_stops_clean = stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]].copy()
gtfs_stops_clean["stop_id"] = gtfs_stops_clean["stop_id"].astype(str)
netex_all["extracted_stop_id"] = netex_all["extracted_stop_id"].astype(str)

# Join on ID
station_match_id = gtfs_stops_clean.merge(
    netex_all[["extracted_stop_id", "stopplace_id", "stopplace_name", "latitude", "longitude"]],
    left_on="stop_id",
    right_on="extracted_stop_id",
    how="left"
)

# Summary
n_gtfs          = station_match_id["stop_id"].nunique()
n_matched       = station_match_id["stopplace_id"].notna().sum()
n_unmatched     = station_match_id["stopplace_id"].isna().sum()
match_rate      = n_matched / n_gtfs * 100

print(f"Total GTFS stops:       {n_gtfs:,}")
print(f"Matched via ID:         {n_matched:,}  ({match_rate:.2f}%)")
print(f"Unmatched:              {n_unmatched:,}")

# Preview matched
print("\n--- Sample matched ---")
display(station_match_id[station_match_id["stopplace_id"].notna()].head(10))

# Preview unmatched 
print("\n--- Sample unmatched ---")
display(station_match_id[station_match_id["stopplace_id"].isna()].head(10))

Total GTFS stops:       2,796
Matched via ID:         2,796  (100.00%)
Unmatched:              0

--- Sample matched ---


,stop_id,stop_name,stop_lat,stop_lon,extracted_stop_id,stopplace_id,stopplace_name,latitude,longitude
0,200403005,"Belair, Sacré-Coeur",49.610276,6.113159,200403005,LU::StopPlace:200403005_CdT::,"Belair, Sacré-Coeur",49.610276,6.113159
1,200402003,"Beggen, Cyprien Merjai",49.643399,6.134779,200402003,LU::StopPlace:200402003_CdT::,"Beggen, Cyprien Merjai",49.643399,6.134779
2,200402004,"Beggen, Henri Dunant",49.647594,6.134546,200402004,LU::StopPlace:200402004_CdT::,"Beggen, Henri Dunant",49.647594,6.134546
3,200304002,"Howald, Bei der Kierch",49.583246,6.142332,200304002,LU::StopPlace:200304002_CdT::,"Howald, Bei der Kierch",49.583246,6.142332
4,200417037,"Kirchberg, Schmëdd",49.628363,6.149234,200417037,LU::StopPlace:200417037_CdT::,"Kirchberg, Schmëdd",49.628363,6.149234
5,200412005,"Gasperich, Clemenceau",49.589887,6.127856,200412005,LU::StopPlace:200412005_CdT::,"Gasperich, Clemenceau",49.589887,6.127856
6,200404022,"Bonnevoie, Kaltreis",49.593658,6.150830,200404022,LU::StopPlace:200404022_CdT::,"Bonnevoie, Kaltreis",49.593658,6.15083
7,200419026,"Limpertsberg, Theater (Bus)",49.617927,6.125584,200419026,LU::StopPlace:200419026_CdT::,"Limpertsberg, Theater (Bus)",49.617927,6.125584
8,200417038,"Kirchberg, Sichegronn",49.628384,6.151532,200417038,LU::StopPlace:200417038_CdT::,"Kirchberg, Sichegronn",49.628384,6.151532
9,200419024,"Limpertsberg, Lefèvre",49.622563,6.108143,200419024,LU::StopPlace:200419024_CdT::,"Limpertsberg, Lefèvre",49.622563,6.108143



--- Sample unmatched ---


,stop_id,stop_name,stop_lat,stop_lon,extracted_stop_id,stopplace_id,stopplace_name,latitude,longitude


## Output

The ID-based join achieved a 100% match rate: all 2,796 GTFS stops found a corresponding NeTEx StopPlace, and the unmatched table is completely empty. Looking at the matched rows, the names are identical on both sides and the coordinates are exactly the same.

## Coordinate Validation

Even though a 100% match rate was achieved I still choose to perform a coordinate validation as a last check.

To confirm the ID-based match, I verified that the coordinates of each matched GTFS stop and its corresponding NeTEx StopPlace agree geographically. For each of the 2,796 matched pairs, I computed the real-world distance in metres between the GTFS (stop_lat, stop_lon) and the NeTEx (latitude, longitude) using the haversine formula. This formula calculates the shortest distance between two points on a sphere given their latitudes and longitudes, making it the standard approach for measuring distances between geographic coordinates.
The calculation was carried out using the haversine Python library (PyPI, 2023), which implements the formula internally.

Sources:

GeeksForGeeks (2022). Haversine formula to find distance between two points on a sphere. https://www.geeksforgeeks.org/dsa/haversine-formula-to-find-distance-between-two-points-on-a-sphere/

PyPI (2023). haversine 2.8.1. https://pypi.org/project/haversine/

In [40]:
# Compute distance in metres between GTFS and NeTEx coordinates for each matched pair
matched = station_match_id[station_match_id["stopplace_id"].notna()].copy()

matched["distance_m"] = matched.apply(
    lambda row: haversine(
        (row["stop_lat"], row["stop_lon"]),
        (float(row["latitude"]), float(row["longitude"])),
        unit=Unit.METERS
    ),
    axis=1
)

# Summary statistics
print("Distance summary (metres):")
print(matched["distance_m"].describe().round(4))

# How many are exactly 0, within 1m, within 10m, and beyond?
print(f"\nExact match (0m):       {(matched['distance_m'] == 0).sum():,}")
print(f"Within 1m:              {(matched['distance_m'] <= 1).sum():,}")
print(f"Within 10m:             {(matched['distance_m'] <= 10).sum():,}")
print(f"Within 50m:             {(matched['distance_m'] <= 50).sum():,}")
print(f"Beyond 50m:             {(matched['distance_m'] > 50).sum():,}")

# Show any outliers beyond 10m — these are worth inspecting
outliers = matched[matched["distance_m"] > 10].sort_values("distance_m", ascending=False)
print(f"\nOutliers beyond 10m: {len(outliers)}")
if len(outliers) > 0:
    display(outliers[["stop_id", "stop_name", "stop_lat", "stop_lon",
                       "latitude", "longitude", "distance_m"]].head(20))

Distance summary (metres):
count    2796.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: distance_m, dtype: float64

Exact match (0m):       2,796
Within 1m:              2,796
Within 10m:             2,796
Within 50m:             2,796
Beyond 50m:             0

Outliers beyond 10m: 0


## Comment 

Every single pair returned a distance of exactly 0.0 metres, with no outliers beyond any threshold. This confirms that the coordinates are not just approximately aligned, they are identical across both feeds.

## Conclusion: Station Comparison

Both feeds contain nearly identical station data: 2,796 GTFS stops and 2,798 NeTEx StopPlaces.

The ID-based matching achieved a 100% match rate, and the coordinate validation confirmed that all matched pairs have a distance of exactly 0.0 metres.

## 2.2 Line level

I now move to the line level.

This is the next comparison level after stops, following the same structure as in the earlier notebooks.

On the GTFS side, `route_short_name` is the main public line label.

On the NeTEx side, the earlier exploration showed that `public_code` is the strongest field for line comparison.

So the next step is to prepare the GTFS and NeTEx line-side tables and compare them at the visible line-label level.

## Parse all Netex files to get the full Line table

In [41]:
all_line_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 100 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for line in root.findall(".//n:Line", ns):
            all_line_rows.append({
                "file_name":      file_name,
                "line_id":        line.get("id"),
                "line_name":      child_text(line, "Name"),
                "short_name":     child_text(line, "ShortName"),
                "public_code":    child_text(line, "PublicCode"),
                "transport_mode": child_text(line, "TransportMode"),
            })

# Raw table before deduplication
netex_lines_raw = pd.DataFrame(all_line_rows)

# Deduplicate on line_id
netex_lines = netex_lines_raw.drop_duplicates(subset=["line_id"]).copy()

print(f"\nRaw Line rows (with duplicates): {len(netex_lines_raw):,}")
print(f"Unique Lines after dedup:        {len(netex_lines):,}")
print(f"\nMissingness after dedup:")
print(netex_lines[["line_name", "short_name", "public_code", "transport_mode"]].isna().sum())

display(netex_lines.head(10))

Total XML files to parse: 700
  0/700 files processed...
  100/700 files processed...
  200/700 files processed...
  300/700 files processed...
  400/700 files processed...
  500/700 files processed...
  600/700 files processed...

Raw Line rows (with duplicates): 700
Unique Lines after dedup:        700

Missingness after dedup:
line_name         0
short_name        0
public_code       0
transport_mode    0
dtype: int64


,file_name,line_id,line_name,short_name,public_code,transport_mode
0,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-10_202604...,LU::Line:257::,10,10,10,bus
1,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-11_202604...,LU::Line:258::,11,11,11,bus
2,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-12_202604...,LU::Line:259::,12,12,12,bus
3,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-13_202604...,LU::Line:260::,13,13,13,bus
4,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-14_202604...,LU::Line:261::,14,14,14,bus
5,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-15_202604...,LU::Line:262::,15,15,15,bus
6,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-16_202604...,LU::Line:263::,16,16,16,bus
7,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-17_202604...,LU::Line:264::,17,17,17,bus
8,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-18_202604...,LU::Line:265::,18,18,18,bus
9,AVL_AVL/NX-PI-01_LU_NAP_LINE_AVL-AVL-19_202604...,LU::Line:266::,19,19,19,bus


## Comment

The full Luxembourg NeTEx line extraction worked cleanly: all 700 XML files were parsed, 700 unique Line objects were obtained, and the main line fields were complete after deduplication.

## Check NeTEx line labels

I now check whether the main visible line labels on the NeTEx side are unique.  
This helps decide whether `public_code` can be used directly for the Luxembourg line comparison.

In [42]:
public_code_counts = (
    netex_lines["public_code"]
    .value_counts(dropna=False)
    .reset_index()
)
public_code_counts.columns = ["public_code", "count"]

repeated_public_codes = public_code_counts[public_code_counts["count"] > 1]

short_name_counts = (
    netex_lines["short_name"]
    .value_counts(dropna=False)
    .reset_index()
)
short_name_counts.columns = ["short_name", "count"]

repeated_short_names = short_name_counts[short_name_counts["count"] > 1]

print("unique public_code:", netex_lines["public_code"].nunique(dropna=True))
print("repeated public_code values:", len(repeated_public_codes))
display(repeated_public_codes.head(20))

print("unique short_name:", netex_lines["short_name"].nunique(dropna=True))
print("repeated short_name values:", len(repeated_short_names))
display(repeated_short_names.head(20))

unique public_code: 690
repeated public_code values: 10


,public_code,count
0,A20,2
1,7,2
2,6,2
3,13,2
4,14,2
5,15,2
6,5,2
7,4,2
8,3,2
9,2,2


unique short_name: 696
repeated short_name values: 4


,short_name,count
0,14,2
1,A20,2
2,15,2
3,13,2


## Comment

The NeTEx line labels are very clean overall.  
`public_code` is almost fully unique, with only 10 repeated values, and `short_name` is even slightly cleaner, with only 4 repeated values.  
This means the Luxembourg line comparison can now be done at the visible line-label level.

## Compare GTFS route_short_name with NeTEx public_code

I now compare the main visible line labels on both sides.  
For GTFS, this is `route_short_name`, and for NeTEx, this is `public_code`.  
Since both sides show a very similar label structure, this is the main Luxembourg line-level comparison.

First, I reduce both datasets to one row per visible line label:
- on the GTFS side, I kept `route_id` and `route_short_name` and deduplicated on `route_short_name`
- on the NeTEx side, I kept `line_id` and `public_code` and deduplicated on `public_code`

Then I merge both tables with an outer join, matching:
- GTFS `route_short_name`
- NeTEx `public_code`

This makes it possible to measure:
- how many labels appear on both sides
- how many appear only in GTFS
- how many appear only in NeTEx
- the resulting match rate on both sides

In [43]:
# GTFS: deduplicate at visible line label level
gtfs_lines_label = (
    routes[["route_id", "route_short_name"]]
    .dropna(subset=["route_short_name"])
    .drop_duplicates(subset=["route_short_name"])
    .copy()
)

# NeTEx: deduplicate at visible line label level
netex_lines_label = (
    netex_lines[["line_id", "public_code"]]
    .dropna(subset=["public_code"])
    .drop_duplicates(subset=["public_code"])
    .copy()
)

# Compare GTFS route_short_name with NeTEx public_code
line_label_compare = gtfs_lines_label.merge(
    netex_lines_label,
    left_on="route_short_name",
    right_on="public_code",
    how="outer",
    indicator=True
)

# Summary
n_gtfs_labels = gtfs_lines_label["route_short_name"].nunique()
n_netex_labels = netex_lines_label["public_code"].nunique()
n_matched_labels = (line_label_compare["_merge"] == "both").sum()
gtfs_match_rate = n_matched_labels / n_gtfs_labels if n_gtfs_labels else 0
netex_match_rate = n_matched_labels / n_netex_labels if n_netex_labels else 0

print("unique GTFS route_short_name:", n_gtfs_labels)
print("unique NeTEx public_code:", n_netex_labels)
print("matched labels:", n_matched_labels)
print("GTFS label match rate:", round(gtfs_match_rate * 100, 2), "%")
print("NeTEx label match rate:", round(netex_match_rate * 100, 2), "%")

print("\nLabels only in GTFS:", (line_label_compare["_merge"] == "left_only").sum())
print("Labels only in NeTEx:", (line_label_compare["_merge"] == "right_only").sum())

display(line_label_compare.head(20))

unique GTFS route_short_name: 690
unique NeTEx public_code: 690
matched labels: 690
GTFS label match rate: 100.0 %
NeTEx label match rate: 100.0 %

Labels only in GTFS: 0
Labels only in NeTEx: 0


,route_id,route_short_name,line_id,public_code,_merge
0,981,1,LU::Line:981::,1,both
1,257,10,LU::Line:257::,10,both
2,258,11,LU::Line:258::,11,both
3,3741,110,LU::Line:3741::,110,both
4,3167,111,LU::Line:3167::,111,both
5,3168,112,LU::Line:3168::,112,both
6,3742,113,LU::Line:3742::,113,both
7,3743,114,LU::Line:3743::,114,both
8,3169,115,LU::Line:3169::,115,both
9,2327,116,LU::Line:2327::,116,both


## Comment

The Luxembourg line-level result is fully consistent.

After deduplication, both sides contain 690 unique visible line labels, and all 690 labels match between GTFS and NeTEx.
There are no labels that appear only in GTFS and none that appear only in NeTEx.


## Conclusion: route level

The Luxembourg route-level comparison gives a very positive result.

On the GTFS side, `route_short_name` was complete and showed 690 unique visible line labels after deduplication.  
On the NeTEx side, `public_code` was also complete and showed the same structure: 690 unique visible line labels.

When both sides were compared at the visible label level, all 690 labels matched.  
There were no labels that appeared only in GTFS and none that appeared only in NeTEx.

So for Luxembourg, the route level shows a full agreement between GTFS and NeTEx at the public line-label level.  
This means the line structure is highly compatible in this dataset and supports the idea of a clear shared denominator between the two formats at this level.

## 2.3 Calendar level

I now move to the calendar level.

As in the earlier notebooks, I first prepare the GTFS side.

The GTFS calendar logic comes from `calendar` and `calendar_dates`, so the next step is to rebuild the active service dates for each `service_id` and summarise them in one pattern table.

In [44]:
# Work on copies
calendar_work = calendar.copy()
calendar_dates_work = calendar_dates.copy()

# Parse dates
calendar_work["start_date"] = pd.to_datetime(calendar_work["start_date"].astype(str), format="%Y%m%d")
calendar_work["end_date"] = pd.to_datetime(calendar_work["end_date"].astype(str), format="%Y%m%d")
calendar_dates_work["date"] = pd.to_datetime(calendar_dates_work["date"].astype(str), format="%Y%m%d")

weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

# Store added and removed dates per service_id
added_dates = defaultdict(set)
removed_dates = defaultdict(set)

for _, row in calendar_dates_work.iterrows():
    sid = row["service_id"]
    d = row["date"]
    if row["exception_type"] == 1:
        added_dates[sid].add(d)
    elif row["exception_type"] == 2:
        removed_dates[sid].add(d)

gtfs_calendar_pattern_rows = []

for _, row in calendar_work.iterrows():
    sid = row["service_id"]
    start = row["start_date"]
    end = row["end_date"]

    all_days = pd.date_range(start, end, freq="D")

    active_dates = set()
    for d in all_days:
        weekday_idx = d.weekday()  # Monday=0 ... Sunday=6
        if row[weekday_cols[weekday_idx]] == 1:
            active_dates.add(d)

    # Apply exceptions
    active_dates = (active_dates - removed_dates[sid]) | added_dates[sid]
    active_dates = sorted(active_dates)

    gtfs_calendar_pattern_rows.append({
        "service_id": sid,
        "start_date": start,
        "end_date": end,
        "n_active_days": len(active_dates),
        "first_active_date": active_dates[0] if active_dates else pd.NaT,
        "last_active_date": active_dates[-1] if active_dates else pd.NaT,
    })

gtfs_calendar_pattern_compare = pd.DataFrame(gtfs_calendar_pattern_rows)

print("GTFS calendar patterns:", len(gtfs_calendar_pattern_compare))
display(gtfs_calendar_pattern_compare.head(20))

GTFS calendar patterns: 100


,service_id,start_date,end_date,n_active_days,first_active_date,last_active_date
0,1,2026-04-08,2026-04-30,14,2026-04-13,2026-04-30
1,2,2026-04-08,2026-04-30,6,2026-04-14,2026-04-30
2,3,2026-04-08,2026-04-30,8,2026-04-13,2026-04-29
3,4,2026-04-08,2026-04-30,8,2026-04-15,2026-04-30
4,5,2026-04-08,2026-04-30,9,2026-04-14,2026-04-30
5,6,2026-04-08,2026-04-30,2,2026-04-17,2026-04-24
6,7,2026-04-08,2026-04-30,12,2026-04-13,2026-04-30
7,8,2026-04-08,2026-04-30,20,2026-04-08,2026-04-30
8,9,2026-04-08,2026-04-30,23,2026-04-08,2026-04-30
9,10,2026-04-08,2026-04-30,3,2026-04-12,2026-04-26


In [45]:
# Sanity check: no service should have 0 active days
print("Services with 0 active days:", (gtfs_calendar_pattern_compare["n_active_days"] == 0).sum())

# And the date range should stay within your feed window
print("Earliest active date:", gtfs_calendar_pattern_compare["first_active_date"].min())
print("Latest active date:",   gtfs_calendar_pattern_compare["last_active_date"].max())

Services with 0 active days: 0
Earliest active date: 2026-04-08 00:00:00
Latest active date: 2026-04-30 00:00:00


## Comment

The GTFS calendar side is now prepared in a comparable pattern table.

A total of 100 GTFS service patterns were rebuilt from `calendar` and `calendar_dates`, with summary fields for the overall validity window and the active service dates.
The sanity check also looks correct: no service has 0 active days, and the active dates stay within the feed window from 2026-04-08 to 2026-04-30.

## Parse all Netex files to get the full Calendar table

In [46]:
all_daytype_rows = []
all_daytypeassignment_rows = []
all_operatingperiod_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 100 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for dt in root.findall(".//n:DayType", ns):
            all_daytype_rows.append({
                "file_name":    file_name,
                "daytype_id":   dt.get("id"),
                "daytype_name": find_text(dt, "n:Name"),
            })

        for dta in root.findall(".//n:DayTypeAssignment", ns):
            all_daytypeassignment_rows.append({
                "file_name":            file_name,
                "assignment_id":        dta.get("id"),
                "order":                find_text(dta, "n:Order"),
                "date":                 find_text(dta, "n:Date"),
                "daytype_ref":          get_ref(dta, "n:DayTypeRef"),
                "operating_period_ref": get_ref(dta, "n:OperatingPeriodRef"),
            })

        for op in root.findall(".//n:UicOperatingPeriod", ns):
            all_operatingperiod_rows.append({
                "file_name":            file_name,
                "operating_period_id":  op.get("id"),
                "from_date":            find_text(op, "n:FromDate"),
                "to_date":              find_text(op, "n:ToDate"),
                "valid_day_bits":       find_text(op, "n:ValidDayBits"),
            })

# Raw tables
netex_daytypes_raw         = pd.DataFrame(all_daytype_rows)
netex_daytypeassignments_raw = pd.DataFrame(all_daytypeassignment_rows)
netex_operatingperiods_raw = pd.DataFrame(all_operatingperiod_rows)

# Deduplicated tables
netex_daytypes         = netex_daytypes_raw.drop_duplicates(subset=["daytype_id"]).copy()
netex_daytypeassignments = netex_daytypeassignments_raw.drop_duplicates(subset=["assignment_id"]).copy()
netex_operatingperiods = netex_operatingperiods_raw.drop_duplicates(subset=["operating_period_id"]).copy()

print("Raw rows:")
print(f"  DayType:            {len(netex_daytypes_raw):,}")
print(f"  DayTypeAssignment:  {len(netex_daytypeassignments_raw):,}")
print(f"  UicOperatingPeriod: {len(netex_operatingperiods_raw):,}")

print("\nUnique after dedup:")
print(f"  DayType:            {len(netex_daytypes):,}")
print(f"  DayTypeAssignment:  {len(netex_daytypeassignments):,}")
print(f"  UicOperatingPeriod: {len(netex_operatingperiods):,}")

print("\nMissingness in UicOperatingPeriod:")
print(netex_operatingperiods[["from_date", "to_date", "valid_day_bits"]].isna().sum())

Total XML files to parse: 700
  0/700 files processed...
  100/700 files processed...
  200/700 files processed...
  300/700 files processed...
  400/700 files processed...
  500/700 files processed...
  600/700 files processed...
Raw rows:
  DayType:            2,042
  DayTypeAssignment:  2,042
  UicOperatingPeriod: 2,042

Unique after dedup:
  DayType:            2,042
  DayTypeAssignment:  2,042
  UicOperatingPeriod: 2,042

Missingness in UicOperatingPeriod:
from_date         0
to_date           0
valid_day_bits    0
dtype: int64


## Comment

The full Luxembourg NeTEx calendar extraction worked cleanly.  
All 700 XML files were parsed, the three calendar-related object types were extracted in equal numbers, no duplicate ids appeared after deduplication, and the main operating-period fields were complete.

## Build the comparable NeTEx calendar pattern table

I now join the NeTEx calendar objects into one comparable pattern table.  
This links `DayTypeAssignment` to `UicOperatingPeriod` and summarises the NeTEx service validity in the same style as the GTFS calendar pattern table.

In [47]:
# Join DayTypeAssignment with UicOperatingPeriod
netex_calendar_pattern_compare = netex_daytypeassignments.merge(
    netex_operatingperiods,
    left_on="operating_period_ref",
    right_on="operating_period_id",
    how="left"
).copy()

# Parse dates
netex_calendar_pattern_compare["from_date"] = pd.to_datetime(
    netex_calendar_pattern_compare["from_date"].str[:10],
    errors="coerce"
)
netex_calendar_pattern_compare["to_date"] = pd.to_datetime(
    netex_calendar_pattern_compare["to_date"].str[:10],
    errors="coerce"
)

# Count active days from valid_day_bits
netex_calendar_pattern_compare["n_active_days"] = (
    netex_calendar_pattern_compare["valid_day_bits"]
    .fillna("")
    .str.count("1")
)

# Derive first and last active date from valid_day_bits
def active_date_bounds(from_date, valid_day_bits):
    if pd.isna(from_date) or pd.isna(valid_day_bits):
        return pd.NaT, pd.NaT

    active_dates = [
        from_date + pd.Timedelta(days=i)
        for i, bit in enumerate(valid_day_bits)
        if bit == "1"
    ]

    if not active_dates:
        return pd.NaT, pd.NaT

    return active_dates[0], active_dates[-1]

bounds = netex_calendar_pattern_compare.apply(
    lambda row: active_date_bounds(row["from_date"], row["valid_day_bits"]),
    axis=1
)

netex_calendar_pattern_compare["first_active_date"] = [b[0] for b in bounds]
netex_calendar_pattern_compare["last_active_date"] = [b[1] for b in bounds]

# Keep a clean comparable view
netex_calendar_pattern_compare = netex_calendar_pattern_compare[
    [
        "daytype_ref",
        "operating_period_ref",
        "from_date",
        "to_date",
        "valid_day_bits",
        "n_active_days",
        "first_active_date",
        "last_active_date",
    ]
].copy()

print("NeTEx calendar patterns:", len(netex_calendar_pattern_compare))
display(netex_calendar_pattern_compare.head(20))

NeTEx calendar patterns: 2042


,daytype_ref,operating_period_ref,from_date,to_date,valid_day_bits,n_active_days,first_active_date,last_active_date
0,LU::DayType:2037::,LU::UicOperatingPeriod:2037::,2026-04-08,2026-04-10,111,3,2026-04-08,2026-04-10
1,LU::DayType:2038::,LU::UicOperatingPeriod:2038::,2026-04-11,2026-04-11,1,1,2026-04-11,2026-04-11
2,LU::DayType:2039::,LU::UicOperatingPeriod:2039::,2026-04-12,2026-04-26,100000010000001,3,2026-04-12,2026-04-26
3,LU::DayType:2040::,LU::UicOperatingPeriod:2040::,2026-04-13,2026-04-30,111110011111001111,14,2026-04-13,2026-04-30
4,LU::DayType:2041::,LU::UicOperatingPeriod:2041::,2026-04-18,2026-04-25,10000001,2,2026-04-18,2026-04-25
5,LU::DayType:2032::,LU::UicOperatingPeriod:2032::,2026-04-11,2026-04-11,1,1,2026-04-11,2026-04-11
6,LU::DayType:2033::,LU::UicOperatingPeriod:2033::,2026-04-08,2026-04-10,111,3,2026-04-08,2026-04-10
7,LU::DayType:2034::,LU::UicOperatingPeriod:2034::,2026-04-12,2026-04-26,100000010000001,3,2026-04-12,2026-04-26
8,LU::DayType:2035::,LU::UicOperatingPeriod:2035::,2026-04-13,2026-04-30,111110011111001111,14,2026-04-13,2026-04-30
9,LU::DayType:2036::,LU::UicOperatingPeriod:2036::,2026-04-18,2026-04-25,10000001,2,2026-04-18,2026-04-25


In [48]:
print("Patterns with missing operating_period_ref:", netex_calendar_pattern_compare["operating_period_ref"].isna().sum())
print("Patterns with 0 active days:", (netex_calendar_pattern_compare["n_active_days"] == 0).sum())
print("Earliest active date:", netex_calendar_pattern_compare["first_active_date"].min())
print("Latest active date:", netex_calendar_pattern_compare["last_active_date"].max())

Patterns with missing operating_period_ref: 0
Patterns with 0 active days: 0
Earliest active date: 2026-04-08 00:00:00
Latest active date: 2026-04-30 00:00:00


## Comment

The NeTEx calendar side is now prepared in a comparable pattern table.

A total of 2,042 NeTEx calendar patterns were rebuilt by linking `DayTypeAssignment` to `UicOperatingPeriod`.
For each pattern, the table now contains the validity window and a summary of the active days through `valid_day_bits`, `n_active_days`, `first_active_date`, and `last_active_date`.

The sanity check also looks correct.
There are no patterns with a missing operating period reference, no patterns with 0 active days, and the active dates stay within the same overall feed window from 2026-04-08 to 2026-04-30.

## Deduplicate the NeTEx calendar side at the pattern level

At the reference level, the NeTEx calendar rows do not need deduplication, because each `daytype_ref` and each `operating_period_ref` is unique.

However, for the actual GTFS–NeTEx comparison, it is still useful to check whether different NeTEx references represent the same calendar pattern.
So the next step is not ID-based deduplication, but pattern-level checking.

In [49]:
pattern_cols = [
    "from_date",
    "to_date",
    "valid_day_bits",
    "n_active_days",
    "first_active_date",
    "last_active_date",
]

netex_pattern_counts = (
    netex_calendar_pattern_compare
    .groupby(pattern_cols)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Total NeTEx calendar rows:        ", len(netex_calendar_pattern_compare))
print("Unique NeTEx pattern signatures:  ", netex_pattern_counts.shape[0])
print("Repeated NeTEx pattern signatures:", (netex_pattern_counts["count"] > 1).sum())

display(netex_pattern_counts.head(20))

Total NeTEx calendar rows:         2042
Unique NeTEx pattern signatures:   115
Repeated NeTEx pattern signatures: 60


,from_date,to_date,valid_day_bits,n_active_days,first_active_date,last_active_date,count
76,2026-04-13,2026-04-30,111110011111001111,14,2026-04-13,2026-04-30,442
29,2026-04-08,2026-04-30,11110111111011111101111,20,2026-04-08,2026-04-30,177
20,2026-04-08,2026-04-30,11100111110011111001111,17,2026-04-08,2026-04-30,174
35,2026-04-08,2026-04-30,11111111111111111111111,23,2026-04-08,2026-04-30,159
62,2026-04-13,2026-04-29,10101001010100101,8,2026-04-13,2026-04-29,120
56,2026-04-12,2026-04-26,100000010000001,3,2026-04-12,2026-04-26,104
82,2026-04-14,2026-04-30,10100001010000101,6,2026-04-14,2026-04-30,102
47,2026-04-11,2026-04-25,100000010000001,3,2026-04-11,2026-04-25,100
49,2026-04-11,2026-04-26,1100000110000011,6,2026-04-11,2026-04-26,90
95,2026-04-17,2026-04-24,10000001,2,2026-04-17,2026-04-24,55


## Comment

The NeTEx calendar rows are unique by reference, but not unique at the pattern level.

Out of 2,042 NeTEx calendar rows, only 115 distinct validity patterns appear.
This means many different NeTEx references still describe the same calendar logic, and some patterns are repeated very often across the dataset.

So for the Luxembourg calendar comparison, the more meaningful unit is the unique pattern signature, not the raw NeTEx row count.

I now reduce the GTFS calendar side to unique pattern signatures as well.

This puts GTFS and NeTEx on the same comparison level:
instead of comparing raw rows, I compare distinct calendar patterns based on validity window and active-day structure.

In [50]:
# GTFS unique pattern signatures
gtfs_pattern_cols = [
    "start_date",
    "end_date",
    "n_active_days",
    "first_active_date",
    "last_active_date",
]

gtfs_pattern_counts = (
    gtfs_calendar_pattern_compare
    .groupby(gtfs_pattern_cols)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Total GTFS calendar rows:        ", len(gtfs_calendar_pattern_compare))
print("Unique GTFS pattern signatures:  ", gtfs_pattern_counts.shape[0])
print("Repeated GTFS pattern signatures:", (gtfs_pattern_counts["count"] > 1).sum())

display(gtfs_pattern_counts.head(20))

Total GTFS calendar rows:         100
Unique GTFS pattern signatures:   88
Repeated GTFS pattern signatures: 9


,start_date,end_date,n_active_days,first_active_date,last_active_date,count
75,2026-04-08,2026-04-30,17,2026-04-08,2026-04-30,3
81,2026-04-08,2026-04-30,19,2026-04-08,2026-04-30,3
83,2026-04-08,2026-04-30,20,2026-04-08,2026-04-30,3
78,2026-04-08,2026-04-30,18,2026-04-08,2026-04-30,2
53,2026-04-08,2026-04-30,9,2026-04-08,2026-04-18,2
61,2026-04-08,2026-04-30,11,2026-04-08,2026-04-19,2
70,2026-04-08,2026-04-30,14,2026-04-13,2026-04-30,2
63,2026-04-08,2026-04-30,11,2026-04-13,2026-04-30,2
67,2026-04-08,2026-04-30,13,2026-04-13,2026-04-30,2
8,2026-04-08,2026-04-30,1,2026-04-26,2026-04-26,1


## Comment

The GTFS calendar rows are also not fully unique at the pattern level.

Out of 100 GTFS calendar rows, 88 distinct pattern signatures appear, and 9 of those signatures are repeated.
So, as on the NeTEx side, the more meaningful comparison unit is the unique calendar pattern rather than the raw row count.

## Comparison of unique GTFS and NeTEx calendar patterns



To compare the calendar data between GTFS and NeTEx, the core challenge is that the two formats encode service validity in fundamentally different ways. GTFS defines services through weekday flags and date exceptions, while NeTEx encodes validity as a binary bit string where each position represents one day in the operating period. To make them directly comparable, I need to bring both sides to the same representation.

On the GTFS side, I rebuild the active service dates for each `service_id` by expanding the weekday flags over the full date range and applying the exceptions from `calendar_dates`. Then I encode the result as a bit string over the full 23-day feed window (April 8–30), where each position is 1 if the service is active on that day and 0 if not.

On the NeTEx side, the `valid_day_bits` string already exists, but it is anchored to each operating period's own from_date, which may differ across records. I re-anchor every bit string to the same feed window start date so that position 0 always means April 8 on both sides.

Once both sides share the same bit string representation over the same window, I deduplicate each to get only truly distinct calendar patterns and perform an outer merge. Two patterns are considered a match only if their bit strings are identical, which means they have the same active days on the same dates, not just the same count or the same start and end date.

In [51]:
# Step 1: rebuild GTFS patterns, storing the bit string
feed_dates = pd.date_range("2026-04-08", "2026-04-30")

gtfs_calendar_pattern_rows = []

for _, row in calendar_work.iterrows():
    sid = row["service_id"]
    start = row["start_date"]
    end = row["end_date"]
    all_days = pd.date_range(start, end, freq="D")

    active_dates = set()
    for d in all_days:
        if row[weekday_cols[d.weekday()]] == 1:
            active_dates.add(d)

    active_dates = (active_dates - removed_dates[sid]) | added_dates[sid]
    active_dates = sorted(active_dates)

    # Generate bit string over the full feed window (1 = active, 0 = not)
    bit_string = "".join("1" if d in set(active_dates) else "0" for d in feed_dates)

    gtfs_calendar_pattern_rows.append({
        "service_id":        sid,
        "start_date":        start,
        "end_date":          end,
        "n_active_days":     len(active_dates),
        "first_active_date": active_dates[0] if active_dates else pd.NaT,
        "last_active_date":  active_dates[-1] if active_dates else pd.NaT,
        "valid_day_bits":    bit_string,
    })

gtfs_calendar_pattern_compare = pd.DataFrame(gtfs_calendar_pattern_rows)

# Step 2: re-anchor NeTEx valid_day_bits to the same feed window
def realign_bits(from_date, valid_day_bits, feed_dates):
    if pd.isna(from_date) or not valid_day_bits:
        return None
    result = []
    for d in feed_dates:
        offset = (d - from_date).days
        if 0 <= offset < len(valid_day_bits):
            result.append(valid_day_bits[offset])
        else:
            result.append("0")
    return "".join(result)

netex_calendar_pattern_compare["bits_aligned"] = netex_calendar_pattern_compare.apply(
    lambda row: realign_bits(row["from_date"], row["valid_day_bits"], feed_dates),
    axis=1
)

# Step 3: deduplicate both sides on the exact fingerprint
compare_cols = ["n_active_days", "first_active_date", "last_active_date", "valid_day_bits"]

gtfs_patterns_unique = (
    gtfs_calendar_pattern_compare[compare_cols]
    .drop_duplicates()
    .copy()
)

netex_patterns_unique = (
    netex_calendar_pattern_compare
    .drop(columns=["valid_day_bits"])                    # drop original bits
    .rename(columns={"bits_aligned": "valid_day_bits"}) # promote aligned bits
    [compare_cols]
    .drop_duplicates()
    .copy()
)

# Step 4: outer merge on exact fingerprint
calendar_pattern_compare = gtfs_patterns_unique.merge(
    netex_patterns_unique,
    on=compare_cols,
    how="outer",
    indicator=True
)

# Step 5: summary
n_gtfs    = len(gtfs_patterns_unique)
n_netex   = len(netex_patterns_unique)
n_matched = (calendar_pattern_compare["_merge"] == "both").sum()

print(f"Unique GTFS patterns:     {n_gtfs}")
print(f"Unique NeTEx patterns:    {n_netex}")
print(f"Matched patterns:         {n_matched}")
print(f"GTFS pattern match rate:  {round(n_matched / n_gtfs * 100, 2)} %")
print(f"NeTEx pattern match rate: {round(n_matched / n_netex * 100, 2)} %")
print(f"\nOnly in GTFS:  {(calendar_pattern_compare['_merge'] == 'left_only').sum()}")
print(f"Only in NeTEx: {(calendar_pattern_compare['_merge'] == 'right_only').sum()}")

display(calendar_pattern_compare.head(20))

Unique GTFS patterns:     100
Unique NeTEx patterns:    115
Matched patterns:         81
GTFS pattern match rate:  81.0 %
NeTEx pattern match rate: 70.43 %

Only in GTFS:  19
Only in NeTEx: 34


,n_active_days,first_active_date,last_active_date,valid_day_bits,_merge
0,1,2026-04-09,2026-04-09,01000000000000000000000,right_only
1,1,2026-04-10,2026-04-10,00100000000000000000000,both
2,1,2026-04-11,2026-04-11,00010000000000000000000,both
3,1,2026-04-12,2026-04-12,00001000000000000000000,both
4,1,2026-04-13,2026-04-13,00000100000000000000000,both
5,1,2026-04-16,2026-04-16,00000000100000000000000,both
6,1,2026-04-17,2026-04-17,00000000010000000000000,both
7,1,2026-04-18,2026-04-18,00000000001000000000000,both
8,1,2026-04-22,2026-04-22,00000000000000100000000,right_only
9,1,2026-04-25,2026-04-25,00000000000000000100000,both


## Comment

Out of 100 GTFS patterns and 115 NeTEx patterns, **81 matched exactly**, giving a **GTFS match rate of 81%** and a **NeTEx match rate of 70.43%**.

With that in mind, an 81% exact match is a good result and suggests the two feeds are broadly consistent on the calendar side.

In [52]:
# How many trips share each GTFS service_id?
trips_per_service = (
    trips.groupby("service_id")
    .size()
    .reset_index(name="n_trips")
    .sort_values("n_trips", ascending=False)
)

print("Trips per service_id:")
print(trips_per_service["n_trips"].describe().round(1))

Trips per service_id:
count     100.0
mean      306.8
std       908.9
min         1.0
25%         2.8
50%        14.0
75%        81.0
max      4904.0
Name: n_trips, dtype: float64


## Observation
### Why GTFS and NeTEx Have Different Numbers of Calendar Patterns

A calendar pattern is simply the answer to: *on which exact days does this service run?*

In GTFS, many services share the same schedule: 30,680 trips are grouped into just 100 patterns. This grouping is very uneven: the median pattern covers only 14 trips, while some cover thousands at once. 

In NeTEx, every service gets its own individual schedule, producing 2,042 raw patterns. But many of those turn out to be identical, so after removing duplicates only **115 truly unique schedules** remain.

The actual comparison was therefore between **100 GTFS schedules** and **115 NeTEx schedules**, of which **81 matched exactly**. The 19 and 34 unmatched patterns on each side are most likely a consequence of this difference in grouping, not a real inconsistency in the data.

## Conclusion: calendar-level

The calendar comparison achieved an **81% exact match rate on the GTFS side** and **70.43% on the NeTEx side**. The unmatched patterns are not errors, instead they reflect a structural difference in how the two formats group service validity. GTFS bundles many trips into shared schedules, while NeTEx assigns an individual calendar to each service. Given this difference, 81% is a great result and confirms that both feeds are broadly consistent at the calendar level.

In [53]:
route_type_labels = {
    0: "Tram",
    1: "Subway / Metro",
    2: "Rail",
    3: "Bus",
    4: "Ferry",
    5: "Cable Tram",
    6: "Aerial Lift",
    7: "Funicular",
    11: "Trolleybus",
    12: "Monorail"
}

routes["route_type_label"] = routes["route_type"].map(route_type_labels)
print(routes["route_type_label"].value_counts())

route_type_label
Bus     694
Rail      5
Tram      1
Name: count, dtype: int64
